# Movie Recommender System

## How to Use This Notebook

This notebook builds a stacked ensemble movie recommender system trained on the MovieLens dataset. 
It combines seven collaborative-filtering algorithms into a single meta-learner and exposes three recommendation engines depending on how much is known about a user.

**Run the blocks in order, top to bottom.** Each block depends on variables and models produced by the ones before it. 
The one exception is Block 13 (test-set evaluation), which is intentionally placed near the end — 
do not inspect or use `test_master` before reaching that block.

### Required files

Place the following four tab-separated files inside a `data/` directory one level above this notebook:

| File | Contents |
|---|---|
| `ratings_train.tsv` | 80,000 user/movie/rating rows used for training |
| `ratings_test.tsv` | 20,000 held-out rows used only in Block 13 |
| `movie_info.tsv` | One row per movie: ID, title, release date, genre flags |
| `user_info.tsv` | One row per user: ID, age, gender, occupation, zip code |

### Block summary

| Block | Purpose |
|---|---|
| 1 | Install and import all dependencies |
| 2 | Load the four raw data files |
| 3–5 | Clean movies, users, and ratings |
| 6 | Join ratings, users, and movies into master tables |
| 7 | Build a popularity-based fallback engine for new users |
| 8 | Build a content-based KNN engine for users with very few ratings |
| 9 | Train seven base models via 10-fold cross-validation, producing out-of-fold (OOF) predictions |
| 10 | Inspect each base model's OOF RMSE; drop weak ones before continuing |
| 11 | Train a Gradient Boosting meta-learner on the OOF predictions |
| 12 | Re-train all base models on the full training set for final inference |
| 13 | Evaluate the ensemble on the held-out test set exactly once |
| 14 | Routing function that picks the right engine for any user |
| 15 | Interactive prompt that accepts demographic preferences and returns personalised recommendations |

### Estimated runtime

Blocks 1–8 complete in under a minute. Block 9 (10-fold training of seven models) 
takes roughly 15–25 minutes depending on hardware. Blocks 10–15 are fast.

## Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from math import sqrt
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import GradientBoostingRegressor
import lightgbm as lgb
from scipy.sparse import csr_matrix
from surprise import SVDpp, SVD, KNNWithMeans, BaselineOnly, SlopeOne, CoClustering, Dataset as SD, Reader
from surprise.model_selection import KFold
print('Imports done')

Imports done


`pandas` and `numpy` handle data manipulation. 
`scikit-learn` provides the KNN model, the Gradient Boosting meta-learner, and the RMSE metric. 
`scipy.sparse` converts the user-movie rating matrix into a memory-efficient sparse format. 
`surprise` is a dedicated collaborative-filtering library that supplies SVD, SVD++, KNN, SlopeOne, Co-Clustering, and Baseline models along with the cross-validation utilities used in Block 9.

**How to use it:** Run this block first. If any import fails, install the missing packages.

## Load Raw Files

In [2]:
ratings_train = pd.read_csv('ratings_train.tsv', sep='\t')
ratings_test  = pd.read_csv('ratings_test.tsv',  sep='\t')
movie_info    = pd.read_csv('movie_info.tsv',    sep='\t')
user_info     = pd.read_csv('user_info.tsv',     sep='\t')
print(f'Train: {len(ratings_train):,} | Test: {len(ratings_test):,} | Movies: {len(movie_info):,} | Users: {len(user_info):,}')

Train: 80,000 | Test: 20,000 | Movies: 1,682 | Users: 943


**What this block does:** Reads the four source files from the `../data/` directory into pandas DataFrames. 
All four files are tab-separated (TSV). The printed summary confirms the expected row counts: 
80,000 training ratings, 20,000 test ratings, 1,682 movies, and 943 users.

**How to use it:** Your files might be in a different directory, update the path strings. 
The test set (`ratings_test`) is loaded here but must not be used for any modelling decision until Block 13.

## Clean Movies

In [3]:
movie_info['release_date'] = pd.to_datetime(movie_info['release_date'], format='%d-%b-%Y', errors='coerce')
movie_info['release_year'] = movie_info['release_date'].dt.year
movie_info['decade']       = (movie_info['release_year'] // 10 * 10).astype('Int64').astype(str)
movie_info = movie_info.dropna(subset=['release_date'])
movie_info = movie_info.rename(columns={'movie_title': 'title'})
genre_cols  = [c for c in movie_info.columns if c.startswith('genre_')]
print(f'Movies cleaned: {len(movie_info):,} | Genres: {len(genre_cols)}')

Movies cleaned: 1,681 | Genres: 19


**What this block does:** Prepares the movie metadata for use throughout the notebook. 
The raw `release_date` string (e.g. `01-Jan-1995`) is parsed into a proper datetime, 
then a numeric `release_year` and a rounded `decade` column are derived from it. 
One movie that has no parseable release date is dropped, leaving 1,681 entries. 
The column `movie_title` is renamed to `title` for consistency, and the 19 binary genre columns 
(those whose names start with `genre_`) are collected into a list for reuse in later blocks.

**How to use it:** Run after Block 2. The `genre_cols` list is referenced in Blocks 7, 14, and 15, 
so this block must complete before those run.

## Clean Users

In [4]:
user_info['age']        = user_info['age'].fillna(user_info['age'].median())
user_info['occupation'] = user_info['occupation'].fillna('other')
print(f'Users cleaned: {len(user_info):,}')
print(user_info.isnull().sum())

Users cleaned: 943
user_id       0
age           0
gender        0
occupation    0
zip_code      0
dtype: int64


**What this block does:** Fills in the two columns of `user_info` that can contain missing values. 
Any missing `age` is replaced with the median age across all users. 
Any missing `occupation` is replaced with the string `'other'`. 
The null-count printout confirms that no missing values remain after these two operations.

**How to use it:** Run after Block 2. No configuration is required. 
If your dataset has different missing-value patterns, adjust the fill strategy here before continuing.

## Clean Ratings and Parse Dates

In [5]:
ratings_train['date'] = pd.to_datetime(ratings_train['date'], errors='coerce')
ratings_test['date']  = pd.to_datetime(ratings_test['date'],  errors='coerce')
print(f'Rating range: {ratings_train["rating"].min()} - {ratings_train["rating"].max()}')
print(f'Train nulls: {ratings_train.isnull().sum().sum()} | Test nulls: {ratings_test.isnull().sum().sum()}')

Rating range: 1 - 5
Train nulls: 0 | Test nulls: 0


**What this block does:** Parses the `date` column in both the training and test rating tables 
into proper datetime objects. It then verifies that ratings fall in the expected 1–5 range 
and that neither table contains null values.

**How to use it:** Run after Block 2. The date column is not used in modelling but may be useful 
for temporal analysis or debugging. The null check is a sanity gate — if it reports any nulls, 
investigate the source files before continuing.

## Build Master Tables

In [6]:
# test_master is built here but NEVER used until Block 13 (evaluation)
train_master = ratings_train.merge(user_info, on='user_id', how='left').merge(movie_info, on='movie_id', how='left')
test_master  = ratings_test.merge(user_info,  on='user_id', how='left').merge(movie_info, on='movie_id', how='left')
print(f'Train master: {len(train_master):,} rows')
print(f'Test master:  {len(test_master):,} rows  (sealed until Block 13)')

Train master: 80,000 rows
Test master:  20,000 rows  (sealed until Block 13)


**What this block does:** Creates two wide tables by joining each ratings table with the user and movie metadata. 
`train_master` is the primary table used by every downstream block for feature computation and model training. 
`test_master` is created here for convenience but is intentionally left untouched until Block 13.

**How to use it:** Run after Blocks 3, 4, and 5. Do not inspect, filter, or derive statistics from 
`test_master` before Block 13 — doing so would leak information from the test set into the model.

## Popularity Engine (Cold Start)

In [7]:
def build_popularity(df, min_ratings=30):
    titles = movie_info[['movie_id','title']].drop_duplicates()
    def top(subset):
        s = subset.groupby('movie_id')['rating'].agg(['count','mean'])
        return s[s['count'] >= min_ratings].reset_index().merge(titles, on='movie_id') \
                .sort_values('mean', ascending=False).head(10)[['title','mean','count']]
    return {'overall': top(df), 'male': top(df[df['gender']=='M']), 'female': top(df[df['gender']=='F'])}

popularity_lists = build_popularity(train_master)
print('Popularity engine ready — Top 5 overall:')
print(popularity_lists['overall'].head().to_string(index=False))

Popularity engine ready — Top 5 overall:
                           title     mean  count
               Casablanca (1942) 4.494898    196
           Close Shave, A (1995) 4.489583     96
         Schindler's List (1993) 4.485944    249
Shawshank Redemption, The (1994) 4.458716    218
           Third Man, The (1949) 4.450980     51


**What this block does:** Builds a simple popularity-based fallback engine used when a user has no rating history. 
It computes three ranked lists from `train_master` — one overall, one for male users, and one for female users — 
each containing the top-10 movies by average rating among films with at least 30 ratings. 
The `min_ratings=30` threshold prevents obscure films with one or two inflated ratings from dominating the list.

**How to use it:** Run after Block 6. The `popularity_lists` dictionary is consumed by the recommendation router 
in Block 14. To raise or lower the minimum-rating threshold, change the `min_ratings` argument when calling 
`build_popularity`.

## KNN Engine (Sparse History)

In [8]:
pivot     = train_master.pivot_table(index='movie_id', columns='user_id', values='rating').fillna(0)
knn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=20)
knn_model.fit(csr_matrix(pivot.values))
titles_idx = movie_info.set_index('movie_id')['title']

def get_knn_recs(movie_id, n=10):
    if movie_id not in pivot.index: return []
    vec  = pivot.iloc[pivot.index.get_loc(movie_id)].values.reshape(1,-1)
    idxs = knn_model.kneighbors(vec, n_neighbors=n+1)[1].flatten()[1:]
    return [titles_idx.get(pivot.index[i], f'Movie {pivot.index[i]}') for i in idxs]

print(f'KNN ready — {pivot.shape[0]} movies x {pivot.shape[1]} users')

KNN ready — 1651 movies x 943 users


**What this block does:** Builds an item-based KNN model used when a user has between 1 and 4 ratings — 
enough to identify taste, but too few for the ensemble to be reliable. 
A movie-by-user pivot matrix is constructed (missing entries filled with zero), 
then a brute-force cosine-similarity KNN model is fitted on the sparse representation of that matrix. 
The `get_knn_recs` function accepts a `movie_id` and returns the titles of the `n` most similar movies 
based on shared rating patterns.

**How to use it:** Run after Block 6. The KNN model is called by the router in Block 14. 
To increase the neighbourhood size, raise `n_neighbors` in the `NearestNeighbors` constructor. 
Note that a larger neighbourhood improves recall but increases memory usage proportionally.

## Train Base Models (Out-of-Fold, 10-Fold)

In [9]:
# Stat features computed PER FOLD from that fold's training data only — no leakage
full_sd = SD.load_from_df(train_master[['user_id','movie_id','rating']], Reader(rating_scale=(1,5)))
base_models = {
    'SVDpp':    SVDpp(n_factors=60,  n_epochs=80, lr_all=0.007, reg_all=0.007, random_state=42),
    'SVD':      SVD(n_factors=200,   n_epochs=60, lr_all=0.005, reg_all=0.015, random_state=42),
    'iKNN':     KNNWithMeans(k=40, sim_options={'name':'pearson_baseline','user_based':False}, verbose=False),
    'uKNN':     KNNWithMeans(k=40, sim_options={'name':'pearson_baseline','user_based':True},  verbose=False),
    'SlopeOne': SlopeOne(),
    'CoClu':    CoClustering(n_epochs=40, random_state=42),
    'Baseline': BaselineOnly(verbose=False),
}
kf       = KFold(n_splits=10, random_state=42, shuffle=True)
oof_rows = []
for fold_num, (trainset_fold, testset_fold) in enumerate(kf.split(full_sd), 1):
    print(f'-- Fold {fold_num}/10 --')
    testset_list = list(testset_fold)
    uids = [x[0] for x in testset_list]
    iids = [x[1] for x in testset_list]
    rs   = [x[2] for x in testset_list]
    # Per-fold stat features — only from this fold's training ratings
    fold_df = pd.DataFrame(
        [(trainset_fold.to_raw_uid(u), trainset_fold.to_raw_iid(i), r)
         for u,i,r in trainset_fold.all_ratings()],
        columns=['user_id','movie_id','rating']
    )
    gm   = fold_df['rating'].mean()
    ustat = fold_df.groupby('user_id')['rating'].agg(['mean','std','count']).fillna(0)
    istat = fold_df.groupby('movie_id')['rating'].agg(['mean','std','count']).fillna(0)
    ustat['bias'] = ustat['mean'] - gm
    istat['bias'] = istat['mean'] - gm
    def sf(uid, mid):
        u = ustat.loc[uid] if uid in ustat.index else None
        i = istat.loc[mid] if mid in istat.index else None
        return [
            u['bias'] if u is not None else 0.0,
            u['std']  if u is not None else 0.0,
            np.log1p(u['count']) if u is not None else 0.0,
            i['bias'] if i is not None else 0.0,
            i['std']  if i is not None else 0.0,
            np.log1p(i['count']) if i is not None else 0.0,
        ]
    fold_preds = {}
    for name, m in base_models.items():
        m.fit(trainset_fold)
        fold_preds[name] = np.clip([m.predict(u,i).est for u,i in zip(uids,iids)], 1, 5)
        print(f'  {name} done')
    for k in range(len(testset_list)):
        row = {name: fold_preds[name][k] for name in base_models}
        row['true'] = rs[k]
        s = sf(uids[k], iids[k])
        row.update({'user_bias':s[0],'user_std':s[1],'user_count':s[2],
                    'item_bias':s[3],'item_std':s[4],'item_count':s[5]})
        oof_rows.append(row)
print('\nOOF training complete')

-- Fold 1/10 --
  SVDpp done
  SVD done
  iKNN done
  uKNN done
  SlopeOne done
  CoClu done
  Baseline done
-- Fold 2/10 --
  SVDpp done
  SVD done
  iKNN done
  uKNN done
  SlopeOne done
  CoClu done
  Baseline done
-- Fold 3/10 --
  SVDpp done
  SVD done
  iKNN done
  uKNN done
  SlopeOne done
  CoClu done
  Baseline done
-- Fold 4/10 --
  SVDpp done
  SVD done
  iKNN done
  uKNN done
  SlopeOne done
  CoClu done
  Baseline done
-- Fold 5/10 --
  SVDpp done
  SVD done
  iKNN done
  uKNN done
  SlopeOne done
  CoClu done
  Baseline done
-- Fold 6/10 --
  SVDpp done
  SVD done
  iKNN done
  uKNN done
  SlopeOne done
  CoClu done
  Baseline done
-- Fold 7/10 --
  SVDpp done
  SVD done
  iKNN done
  uKNN done
  SlopeOne done
  CoClu done
  Baseline done
-- Fold 8/10 --
  SVDpp done
  SVD done
  iKNN done
  uKNN done
  SlopeOne done
  CoClu done
  Baseline done
-- Fold 9/10 --
  SVDpp done
  SVD done
  iKNN done
  uKNN done
  SlopeOne done
  CoClu done
  Baseline done
-- Fold 10/10 --
  

**What this block does:** This is the core training step. Seven collaborative-filtering algorithms are trained 
using 10-fold cross-validation to produce out-of-fold (OOF) predictions — that is, predictions made on 
each example by a model that was never trained on that example. This is the standard technique for building 
a stacking ensemble without data leakage.

The seven base models are:

| Name | Algorithm |
|---|---|
| SVDpp | SVD with implicit feedback (more expressive but slower) |
| SVD | Standard matrix factorisation |
| iKNN | Item-based KNN with Pearson baseline similarity |
| uKNN | User-based KNN with Pearson baseline similarity |
| SlopeOne | Weighted slope-one interpolation |
| CoClu | Co-clustering of users and items |
| Baseline | Global mean plus user and item bias terms |

In addition to each model's prediction, six statistical features are computed per fold from the fold's 
own training data: user rating bias, user rating standard deviation, log-count of user ratings, 
and the same three features for the item. These are appended as extra columns for the meta-learner.

**How to use it:** Run after Block 6. This block takes the most time — budget 15–25 minutes. 
Do not change `random_state` values mid-run; doing so will invalidate the OOF predictions. 
If you want to remove a base model from the ensemble, delete it from the `base_models` dictionary 
before running this block (not after).

## Inspect OOF Individual RMSEs

In [10]:
# Run this before training the meta-learner.
# If any model's OOF RMSE exceeds 0.97, it is likely weakening the ensemble — remove it from base_models.
oof_df    = pd.DataFrame(oof_rows)
feat_cols = list(base_models.keys()) + ['user_bias','user_std','user_count','item_bias','item_std','item_count']
X_oof     = oof_df[feat_cols].values
y_oof     = oof_df['true'].values

print('OOF individual RMSEs (lower = better base model):')
for name in base_models:
    r = sqrt(mean_squared_error(y_oof, oof_df[name].values))
    flag = '  (consider dropping)' if r > 0.97 else ''
    print(f'  {name:10s}: {r:.4f}{flag}')

OOF individual RMSEs (lower = better base model):
  SVDpp     : 1.0755  (consider dropping)
  SVD       : 0.9725  (consider dropping)
  iKNN      : 0.9294
  uKNN      : 0.9437
  SlopeOne  : 0.9493
  CoClu     : 0.9707  (consider dropping)
  Baseline  : 0.9464


**What this block does:** Computes the RMSE of each base model individually using the OOF predictions 
collected in Block 9. This gives a quick read on which models are pulling their weight in the ensemble. 
Models with an RMSE above 0.97 are flagged as candidates for removal, because a weak base model can 
add noise rather than signal to the meta-learner.

**How to use it:** Run after Block 9. Read the output carefully before continuing. 
If one or more models are flagged, go back to the `base_models` dictionary in Block 9, remove those entries, 
and re-run Blocks 9 and 10. Only proceed to Block 11 once you are satisfied with the lineup. 
Note that the meta-learner can sometimes compensate for a weak base model by down-weighting it, 
so removing it is a judgement call rather than a hard rule.

## Train Meta-Learner (Gradient Boosting)

In [11]:
meta = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=30,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
meta.fit(X_oof, y_oof)
oof_blend = np.clip(meta.predict(X_oof), 1, 5)
print(f'GBM meta-learner OOF RMSE: {sqrt(mean_squared_error(y_oof, oof_blend)):.4f}')
print('   (This should be close to the test RMSE — if the gap exceeds 0.01, there may be residual leakage)')

GBM meta-learner OOF RMSE: 0.8639
   (This should be close to the test RMSE — if the gap exceeds 0.01, there may be residual leakage)


**What this block does:** Trains a Gradient Boosting Regressor as the second-level model (meta-learner). 
Its input features are the seven OOF predictions from the base models plus the six statistical features 
computed in Block 9. Its target is the true rating. 
The meta-learner learns how to combine the base models' outputs optimally, including when to trust one 
model more than another based on statistical context (e.g. whether the user has many or few ratings).

The hyperparameters — `max_depth=3`, `subsample=0.6`, `min_samples_leaf=50` — are intentionally conservative 
to prevent the meta-learner from overfitting to the OOF predictions.

**How to use it:** Run after Block 10. The OOF RMSE printed here is your best estimate of test performance. 
If this number is significantly lower than the RMSE you see in Block 13, there is likely a form of leakage 
somewhere earlier in the pipeline.

## Re-train Base Models on Full Training Data

In [12]:
full_trainset = full_sd.build_full_trainset()
for name, m in base_models.items():
    m.fit(full_trainset)
    print(f'{name} re-trained on full training set')

# Stat features for inference — derived from full training data
gm_full    = train_master['rating'].mean()
ustat_full = train_master.groupby('user_id')['rating'].agg(['mean','std','count']).fillna(0)
istat_full = train_master.groupby('movie_id')['rating'].agg(['mean','std','count']).fillna(0)
ustat_full['bias'] = ustat_full['mean'] - gm_full
istat_full['bias'] = istat_full['mean'] - gm_full

def get_sf(uid, mid):
    u = ustat_full.loc[uid] if uid in ustat_full.index else None
    i = istat_full.loc[mid] if mid in istat_full.index else None
    return [
        u['bias'] if u is not None else 0.0,
        u['std']  if u is not None else 0.0,
        np.log1p(u['count']) if u is not None else 0.0,
        i['bias'] if i is not None else 0.0,
        i['std']  if i is not None else 0.0,
        np.log1p(i['count']) if i is not None else 0.0,
    ]

def ensemble_predict(uid, mid):
    model_preds = np.clip([m.predict(uid,mid).est for m in base_models.values()], 1, 5)
    x = np.concatenate([model_preds, get_sf(uid, mid)]).reshape(1,-1)
    return float(np.clip(meta.predict(x)[0], 1, 5))

class EnsembleModel:
    def predict(self, uid, mid):
        class P: pass
        p = P(); p.est = ensemble_predict(uid, mid); return p

svd_model = EnsembleModel()
print('\nEnsemble ready for inference')

SVDpp re-trained on full training set
SVD re-trained on full training set
iKNN re-trained on full training set
uKNN re-trained on full training set
SlopeOne re-trained on full training set
CoClu re-trained on full training set
Baseline re-trained on full training set

Ensemble ready for inference


**What this block does:** In Block 9 the base models were trained on 90% of the data during each fold. 
Now that the meta-learner is fixed, all base models are re-trained on the complete 80,000-row training set 
so that they have seen as much data as possible before being asked to predict on unseen users and movies.

The block also computes the statistical features (`get_sf`) using full-training statistics — the same six 
features used during OOF training — and assembles the `ensemble_predict` function that routes a 
(user_id, movie_id) pair through all base models, appends the stat features, and returns the meta-learner's 
final prediction clipped to the valid 1–5 range.

**How to use it:** Run after Block 11. After this block completes, `ensemble_predict` and `svd_model` are 
available for both Block 13 evaluation and Block 14 recommendations. Do not re-run this block after 
running Block 13, as that would cause the base models to be re-fitted and could change results.

## Evaluate on Test Set (Run Once, at the End)

In [13]:
# This is the only place test_master is used.
trained_users  = set(train_master['user_id'].unique())
trained_movies = set(train_master['movie_id'].unique())
test_known = test_master[
    test_master['user_id'].isin(trained_users) &
    test_master['movie_id'].isin(trained_movies)
].copy()

print(f'Total test ratings:   {len(test_master):,}')
print(f'Known u+m pairs:      {len(test_known):,}')
print(f'Cold-start (skipped): {len(test_master)-len(test_known):,}')

preds = np.clip(
    [ensemble_predict(uid, mid) for uid, mid in zip(test_known['user_id'], test_known['movie_id'])],
    1, 5
)
rmse = sqrt(mean_squared_error(test_known['rating'], preds))
print(f'\nTest RMSE: {rmse:.4f}')
print('  Target: < 0.901  |', 'Hit' if rmse < 0.901 else 'Miss')

Total test ratings:   20,000
Known u+m pairs:      19,964
Cold-start (skipped): 36

Test RMSE: 0.8993
  Target: < 0.901  | Hit


**What this block does:** Computes the final held-out RMSE on the 20,000-row test set. 
Before predicting, it filters out the small number of test rows where either the user or the movie 
was never seen during training (cold-start cases the ensemble cannot handle). 
Predictions are clipped to [1, 5] and compared against the true ratings using RMSE.

**How to use it:** Run this block exactly once, after Block 12. 
This is the only legitimate use of `test_master`. If the test RMSE is notably higher than the OOF RMSE 
reported in Block 11, revisit the pipeline for sources of leakage. 
Do not use the test RMSE to make further modelling decisions — at that point the test set is no longer truly held-out.

## Recommendation Router

In [14]:
titles_map    = movie_info.set_index('movie_id')['title'].to_dict()
all_movie_ids = train_master['movie_id'].unique().tolist()

def recommend(user_id, n=10):
    seen         = set(train_master[train_master['user_id']==user_id]['movie_id'])
    rating_count = len(seen)
    print(f'\nUser {user_id}  |  {rating_count} ratings  |  ', end='')

    if rating_count == 0:
        print('Engine: Popularity')
        g   = user_info[user_info['user_id']==user_id]['gender'].values
        key = ('male' if g[0]=='M' else 'female') if len(g) else 'overall'
        df  = popularity_lists.get(key, popularity_lists['overall'])
        [print(f'  {i:2}. {r["title"]}  (avg {r["mean"]:.2f})') for i,(_, r) in enumerate(df.head(n).iterrows(), 1)]
        return list(df.head(n)['title'])

    elif rating_count < 5:
        print('Engine: Item KNN')
        recs, top = [], train_master[train_master['user_id']==user_id].sort_values('rating', ascending=False)['movie_id']
        for mid in top:
            for t in get_knn_recs(mid, n=n+10):
                row = movie_info[movie_info['title']==t]
                if not row.empty and row.iloc[0]['movie_id'] not in seen: recs.append(t)
            if len(recs) >= n: break
        recs = list(dict.fromkeys(recs))[:n]
        [print(f'  {i:2}. {t}') for i,t in enumerate(recs,1)]
        return recs

    else:
        print('Engine: Ensemble')
        scored = sorted([(mid, ensemble_predict(user_id,mid)) for mid in all_movie_ids if mid not in seen], key=lambda x:-x[1])
        recs   = [titles_map.get(mid, f'Movie {mid}') for mid,_ in scored[:n]]
        [print(f'  {i:2}. {t}') for i,t in enumerate(recs,1)]
        return recs

recommend(train_master['user_id'].value_counts().index[0])
recommend(train_master['user_id'].value_counts().index[-1])


User 405  |  595 ratings  |  Engine: Ensemble
   1. Raiders of the Lost Ark (1981)
   2. It's a Wonderful Life (1946)
   3. Fargo (1996)
   4. E.T. the Extra-Terrestrial (1982)
   5. Toy Story (1995)
   6. Good Will Hunting (1997)
   7. Return of the Jedi (1983)
   8. Butch Cassidy and the Sundance Kid (1969)
   9. Die Hard (1988)
  10. Tomorrow Never Dies (1997)

User 364  |  12 ratings  |  Engine: Ensemble
   1. Titanic (1997)
   2. Schindler's List (1993)
   3. Good Will Hunting (1997)
   4. Usual Suspects, The (1995)
   5. Star Wars (1977)
   6. Raiders of the Lost Ark (1981)
   7. Shawshank Redemption, The (1994)
   8. Bad Taste (1987)
   9. Apollo 13 (1995)
  10. Hugo Pool (1997)


['Titanic (1997)',
 "Schindler's List (1993)",
 'Good Will Hunting (1997)',
 'Usual Suspects, The (1995)',
 'Star Wars (1977)',
 'Raiders of the Lost Ark (1981)',
 'Shawshank Redemption, The (1994)',
 'Bad Taste (1987)',
 'Apollo 13 (1995)',
 'Hugo Pool (1997)']

**What this block does:** Defines the `recommend(user_id, n)` function, which is the single entry point 
for generating recommendations. It selects one of three engines based on how many ratings the user has:

- **0 ratings** — uses the popularity engine from Block 7, filtered by the user's gender if available.
- **1–4 ratings** — uses the KNN engine from Block 8, finding movies similar to the ones the user has rated highly.
- **5 or more ratings** — uses the full ensemble from Block 12, scoring every unseen movie and returning the top `n`.

Movies the user has already rated are excluded from all three engines. 
The two calls at the bottom of the block demonstrate the function on the most-active and least-active users in the training set.

**How to use it:** Run after Block 12 (or Block 13 if you have already evaluated). 
Call `recommend(user_id)` with any integer user ID from the dataset. 
Pass `n=` to control the number of results. The ensemble path is slow (it scores every movie), 
so for bulk predictions consider vectorising or caching the results.

## Interactive Recommender

In [ ]:
def ask(prompt, valid=None, cast=str):
    while True:
        try:
            val = cast(input(prompt).strip())
            if valid is None or val in valid: return val
            print(f'  Choose from: {valid}')
        except ValueError: print('  Invalid input, try again.')

def interactive_recommend():
    print('='*50 + '\n        MOVIE RECOMMENDER\n' + '='*50)

    gender     = ask('\nGender (M/F): ', ['M','F'], lambda x: x.upper())
    age        = ask('Age: ', cast=int)
    occs       = sorted(user_info['occupation'].dropna().unique())
    print('\nOccupations:'); [print(f'  {i:2}. {o}') for i,o in enumerate(occs,1)]
    occupation = occs[ask(f'Pick (1-{len(occs)}): ', list(range(1,len(occs)+1)), int) - 1]

    glabels = [g.replace('genre_','') for g in genre_cols]
    print('\nGenres:'); [print(f'  {i:2}. {g}') for i,g in enumerate(glabels,1)]
    while True:
        try:
            picks = [int(x) for x in input('Pick up to 3 genres (e.g. 1,5,12): ').split(',')]
            if picks and len(picks)<=3 and all(1<=p<=len(glabels) for p in picks):
                fav_genres = [genre_cols[p-1] for p in picks]; break
            print('  Enter 1-3 valid numbers.')
        except ValueError: print('  Use comma-separated numbers.')

    print('\nEra: 1=Classic(<1970)  2=Retro(70s-80s)  3=Modern(90s-00s)  4=Any')
    preferred_decades = {'1':[1930,1940,1950,1960],'2':[1970,1980],'3':[1990,2000],'4':None}[
        ask('Choice (1/2/3/4): ', ['1','2','3','4'])
    ]
    n = ask('How many recommendations? (1-20): ', list(range(1,21)), int)

    print('\nMatching taste profile...')
    cands = user_info[user_info['gender']==gender].copy()
    cands['age_score'] = 1/(1+abs(cands['age']-age))
    cands['occ_score'] = (cands['occupation']==occupation).astype(int)
    genre_means = (
        train_master[train_master['user_id'].isin(cands['user_id'])]
        .assign(fav=lambda d: d[fav_genres].max(axis=1))
        .query('fav==1')
        .groupby('user_id')['rating'].mean()
    )
    cands['genre_score'] = cands['user_id'].map(genre_means).fillna(0)
    for col in ['age_score','occ_score','genre_score']:
        r = cands[col].max()-cands[col].min()
        if r > 0: cands[col] = (cands[col]-cands[col].min())/r
    best_uid = int(cands.assign(total=lambda d: d['age_score']*.3+d['occ_score']*.2+d['genre_score']*.5)
                   .sort_values('total',ascending=False).iloc[0]['user_id'])
    print(f'Matched to user {best_uid}')

    recs = recommend(best_uid, n=n*3)
    if preferred_decades:
        filtered = [t for t in recs
                    if not (r:=movie_info[movie_info['title']==t]).empty
                    and not pd.isna(r.iloc[0]['release_year'])
                    and int(r.iloc[0]['release_year'])//10*10 in preferred_decades]
        recs = filtered if len(filtered)>=n else recs
    recs = recs[:n]

    print(f'\n{"="*50}\n  YOUR TOP {n} RECOMMENDATIONS\n{"="*50}')
    for i,title in enumerate(recs,1):
        row = movie_info[movie_info['title']==title]
        if not row.empty:
            r      = row.iloc[0]
            genres = ', '.join(g.replace('genre_','') for g in genre_cols if r[g]==1)[:40]
            year   = int(r['release_year']) if not pd.isna(r['release_year']) else '?'
            print(f'\n  {i:2}. {title}\n      {year}  |  {genres}')
    print('='*50)
    return recs

results = interactive_recommend()

**What this block does:** Provides a terminal-based interactive interface for users who are not already 
in the dataset. It collects five inputs — gender, age, occupation, up to three favourite genres, and a 
preferred era — then finds the existing user in the training set whose profile most closely matches. 
Matching is a weighted score: genre affinity (50%), age proximity (30%), and occupation match (20%). 
The closest-matching user's ID is passed to `recommend`, and the returned list is optionally filtered 
to the preferred era before being printed with year and genre metadata.

**How to use it:** Run after Block 14. The block will pause and prompt you for input in the output area. 
Enter your responses and press Enter after each one. Genre and era selections narrow the results without 
replacing the underlying model — if the filtered list is shorter than the requested count, 
the unfiltered ranked list is used as a fallback. The final recommendations are also stored in `results` 
for any further use in the notebook.